Copyright 2026 Google LLC

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

<a target="_blank" href="https://colab.research.google.com/github/google-deepmind/gbench/blob/main/examples/notebooks/05_agentic_quality_with_gemmaclaw.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# gbench agentic quality simulation with GemmaClaw

**Author:** [Luciano Martins](https://github.com/lucianommartins)

This notebook demonstrates how to evaluate multi-turn autonomous agent workflows using `gbench --quality-only` and the GemmaClaw QA suite (the QA scenarios of the pinned GemmaClaw commit, 60 at d8bc6989; scenarios that need another model provider are listed apart and not scored). You will learn how `gbench` builds and runs the suite in its local `gbench-quality` Docker image (built once per GemmaClaw commit: several minutes and about 8 GB the first time), and how it tests session recall and tool routing against an OpenAI-compatible `/v1` endpoint.

> **Prerequisites:** the quality pillar needs Docker (the CLI and a reachable daemon), git and a Gemini API key (`GEMINI_API_KEY` or `GEMINI_API_KEYS`). Colab has no Docker daemon, so run the quality cells on a Docker-capable machine; see the prerequisites cell below.

## Serving paths (this notebook shows both)

* **Ollama local serving (featured here):** serve the `gemma-4` QAT GGUF with Ollama and point `gbench` at its OpenAI endpoint `http://localhost:11434/v1`. All cells below use this path.
* **vLLM remote endpoint (alternative):** if you already serve `gemma-4` with vLLM, skip the Ollama setup and run `gbench` against it directly, e.g. `--remote-endpoint http://127.0.0.1:8000/v1 --tokenizer google/gemma-4-26B-A4B-it` (served model id `google/gemma-4-26B-A4B-it`; the agent's tool calls need vLLM's tool-call flags: `vllm serve google/gemma-4-26B-A4B-it --reasoning-parser gemma4 --enable-auto-tool-choice --tool-call-parser gemma4`).

## Learning objectives

1. Configure Ollama to serve a quantized Google Gemma 4 model (`unsloth/gemma-4-E4B-it-qat-GGUF`) with a context window of 32768 tokens (GemmaClaw sends up to 8192 output tokens per request, so the window must hold that plus each scenario's growing conversation).
2. Understand how the QA suite runs in gbench's local `gbench-quality` Docker image, built once per GemmaClaw commit.
3. Execute multi-turn agentic simulation scenarios (`--quality-only`) over OpenAI-compatible `/v1` endpoints.
4. Filter specific capability scenarios by scenario id, for example `--scenarios memory-recall mcp-plugin-tools-call`.
5. Perform a clean session shutdown to terminate background servers and reclaim hardware memory.

## Useful resources

* [gbench GitHub repository](https://www.github.com/google-deepmind/gbench)
* [Ollama documentation](https://github.com/ollama/ollama)
* [GemmaClaw agent simulation harness](https://github.com/gemmaclaw/gemmaclaw)

## 1. Environment setup and installation

We use the `gbench` checkout this notebook is in (or clone the repository from GitHub), change directory into the project root, and install the package in editable mode plus `transformers` (`pip install -e . transformers`), the remote-endpoint install in the [installation guide](../../docs/installation.md). The quality run itself reads only the model's `config.json` (named by `--tokenizer`) from the Hugging Face Hub. This builds and links the `gbench` CLI executable without installing development tools. The [installation guide](../../docs/installation.md) lists every optional install.

In [ ]:
import os, sys
from pathlib import Path

# Safe environment setup: Always normalize to top-level repository
if Path("/content").exists():
    %cd -q /content
    if not Path("/content/gbench").is_dir():
        !git clone https://github.com/google-deepmind/gbench.git
    %cd -q /content/gbench
else:
    # Use the gbench checkout you are in (also when this notebook is opened from
    # examples/notebooks/), else clone one next to it.
    root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "pyproject.toml").is_file() and (p / "gbench" / "cli.py").is_file()), None)
    if root is None:
        if not Path("gbench/pyproject.toml").is_file():
            !git clone https://github.com/google-deepmind/gbench.git
        root = Path("gbench").resolve()
    %cd -q {root}

%pip install -e . transformers -q
import gbench
print(f"gbench version {gbench.__version__} installed successfully.")

# Inspect available evaluation pillars
!gbench --list pillars

## Hugging Face authentication (required)

This notebook downloads the Gemma 4 GGUF (and its vision projector) plus the `google/gemma-4-*` tokenizer from the Hugging Face Hub with `huggingface_hub`. Some are **gated**, so an **`HF_TOKEN` is required**.

1. Create a **read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and accept the license on any gated model page you use.
2. On **Colab**: click the **key icon (Secrets)** in the left sidebar, then **Add new secret**, name it `HF_TOKEN`, paste the token, and toggle **Notebook access** on.
3. **Elsewhere**: set it in your environment, e.g. `export HF_TOKEN=hf_...` (or `os.environ["HF_TOKEN"] = "hf_..."`).

The next cell loads the token and stops with instructions if it is missing.

In [ ]:
import os

# HF_TOKEN is REQUIRED: this notebook downloads the Gemma 4 GGUF (+ vision projector)
# and the google/gemma-4-* tokenizer from the Hugging Face Hub via huggingface_hub,
# which authenticates with it (resumable, higher rate limits, and access to gated repos).
try:
    from google.colab import userdata          # Colab: read from the Secrets vault
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass                                        # not on Colab, or the secret is unset
if not os.environ.get("HF_TOKEN"):
    raise RuntimeError(
        "HF_TOKEN is not set. On Colab: click the key icon (Secrets) in the left "
        "sidebar, add a secret named HF_TOKEN, and turn on notebook access. "
        "Elsewhere: os.environ['HF_TOKEN'] = 'hf_...'. "
        "Create a read token at https://huggingface.co/settings/tokens."
    )
print("HF_TOKEN loaded.")

## Quality pillar prerequisites (Docker and a Gemini key)

The quality pillar runs GemmaClaw's QA suite in gbench's local `gbench-quality` Docker image, so it needs:

* **Docker**: the `docker` CLI and a daemon your user can reach. Colab has no Docker daemon, so run the quality cells on a Docker-capable machine (your workstation or a cloud VM), not in Colab.
* **git**, to fetch the pinned GemmaClaw commit.
* **A Gemini API key** in `GEMINI_API_KEY` (or a comma-separated list in `GEMINI_API_KEYS`): the suite's alternate, image-generation and embedding models run on the Gemini API.

The first quality run builds the image (several minutes, about 8 GB); later runs reuse it. The next cell checks these and stops with instructions if one is missing.

In [ ]:
import os, shutil, subprocess

# Gemini key: Colab Secrets first, else the environment.
try:
    from google.colab import userdata
    for _name in ("GEMINI_API_KEYS", "GEMINI_API_KEY"):
        try:
            _value = userdata.get(_name)
        except Exception:
            _value = None
        if _value:
            os.environ[_name] = _value
except Exception:
    pass
if not (os.environ.get("GEMINI_API_KEYS") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError(
        "Set GEMINI_API_KEY (or GEMINI_API_KEYS) before the quality run: on Colab add it as a "
        "secret with notebook access, elsewhere os.environ['GEMINI_API_KEY'] = '...'.")
if not shutil.which("git"):
    raise RuntimeError("Install git (e.g. apt-get install -y git), then re-run this cell.")
if not shutil.which("docker") or subprocess.run(["docker", "info"], capture_output=True).returncode != 0:
    raise RuntimeError(
        "The quality pillar needs the docker CLI and a daemon this user can reach. Colab has "
        "none: run this notebook on a Docker-capable machine, then re-run this cell.")
print("Docker, git and a Gemini key are available.")

## 2. Installing Ollama locally

We check if the Ollama binary is present on the system. If it is not found, we install Ollama using its official Linux installation script (`curl -fsSL https://ollama.com/install.sh | sh`). Finally, we run `ollama --version` to verify that the installation succeeded and the CLI is available.

In [ ]:
import subprocess, os, shutil, glob

# (Re)install Ollama unless both the binary and its llama-server runner are present.
def _ollama_ready():
    if not shutil.which("ollama"):
        return False
    return any(glob.glob(p) for p in (
        "/usr/local/lib/ollama/llama-server",
        "/usr/local/lib/ollama/*/llama-server",
        "/usr/lib/ollama/llama-server",
    ))

if not _ollama_ready():
    print("Installing/repairing Ollama (binary + llama-server runner)...")
    # Ensure zstd is available (required by Ollama Linux tar.zst packages)
    subprocess.run("command -v zstd >/dev/null || (command -v apt-get >/dev/null && apt-get update -qq && apt-get install -y -qq zstd)", shell=True)
    # Run official Ollama installer
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True)
else:
    print("Ollama (with llama-server runner) already installed.")

# Ensure binary directory is present in PATH for subsequent cells
for p in ["/usr/local/bin", "/usr/bin", os.path.expanduser("~/.local/bin")]:
    if os.path.exists(os.path.join(p, "ollama")) and p not in os.environ.get("PATH", ""):
        os.environ["PATH"] = f"{p}:{os.environ.get('PATH', '')}"

!ollama --version

## 3. Launching background Ollama server

We launch the `ollama serve` process in the background and send a health check request to `http://localhost:11434/` to verify that the HTTP API is alive ("Ollama is running").

In [ ]:
import subprocess, time, requests
try:
    resp = requests.get("http://localhost:11434/", timeout=2)
    print("Ollama server already active:", resp.text.strip())
except Exception:
    print("Starting background ollama serve...")
    OLLAMA_PROC = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL,
                                   stderr=subprocess.DEVNULL)   # stopped in the cleanup cell
    time.sleep(4)
    resp = requests.get("http://localhost:11434/")
    print("Server health check:", resp.text.strip())

## 4. Downloading the GGUF and writing the Modelfile

We download the quantized Gemma 4 GGUF **and its vision projector (`mmproj`)** from the Hugging Face Hub with `huggingface_hub` (authenticated via `HF_TOKEN`, so the download is resumable and not rate-limited), then write an Ollama `Modelfile.qat` that points `FROM` the **local** files:
* **`num_ctx 32768`**: context window of 32768 tokens. GemmaClaw sends up to 8192 output tokens per request, so the window must hold that plus each scenario's conversation. Ollama does not report its window to gbench, so the run cells pass the same value as `--max-model-len 32768`, which gbench declares to GemmaClaw.

The two `FROM` lines (main GGUF + `mmproj`) keep the model's **vision** capability. If `ollama create` in the next cell reports a missing file, re-run this cell.

In [ ]:
import os
from huggingface_hub import HfApi, hf_hub_download

HF_REPO = "unsloth/gemma-4-E4B-it-qat-GGUF"
HF_QUANT = "UD-Q4_K_XL"
token = os.environ["HF_TOKEN"]  # required; loaded in the Hugging Face auth cell above

# Download the model GGUF and its vision projector (mmproj) with your HF_TOKEN, then
# build the Modelfile from the local files (the second FROM line keeps vision).
files = HfApi().list_repo_files(HF_REPO, token=token)
main = [f for f in files if f.endswith(".gguf") and HF_QUANT in f]
proj = [f for f in files if f.endswith(".gguf") and "mmproj" in f.lower() and "-F16" in f]
if not main:
    raise RuntimeError(f"No {HF_QUANT} .gguf found in {HF_REPO}.")
GGUF_PATH = os.path.realpath(hf_hub_download(HF_REPO, main[0], token=token))
lines = [f"FROM {GGUF_PATH}"]
if proj:  # vision projector -> keeps multimodal capability
    lines.append(f"FROM {os.path.realpath(hf_hub_download(HF_REPO, proj[0], token=token))}")
lines += ['PARAMETER num_ctx 32768']
with open("Modelfile.qat", "w", encoding="utf-8") as f:
    f.write("\n".join(lines) + "\n")
print("Created Modelfile.qat from local GGUF" + (" + mmproj (vision)" if proj else ""))

## 5. Registering model and running generation smoke test

We register our custom model tag (`gemma4-qat:4b`) with `ollama create -f Modelfile.qat`. Because `Modelfile.qat` points `FROM` the local GGUF (and `mmproj`) downloaded in the previous cell, this reads from disk, with no network pull. We then run a quick generation test to verify the model loads into hardware memory and generates tokens correctly.

In [ ]:
import subprocess, requests

MODEL_TAG = "gemma4-qat:4b"
print(f"Registering model {MODEL_TAG} from the local GGUF (built in the previous cell)...")
subprocess.run(["ollama", "create", MODEL_TAG, "-f", "Modelfile.qat"], check=True)

print("Running quick generation smoke test via Ollama API (cold load into GPU VRAM)...")
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={"model": MODEL_TAG, "prompt": "Reply with the single word: READY.", "stream": False},
    timeout=300,
)
print("Smoke test response:", resp.json().get("response", "").strip())

## 6. Verifying OpenAI REST endpoint readiness

Before launching `gbench`, we query `http://localhost:11434/v1/models` to verify that Ollama is serving standard OpenAI `/v1` REST payloads and that our registered model is listed.

In [ ]:
import requests
resp = requests.get("http://localhost:11434/v1/models")
print("OpenAI /v1/models endpoint HTTP status:", resp.status_code)
models = [m["id"] for m in (resp.json().get("data") or [])]
print("Available REST models:", models)
if not models:
    print("No models registered yet - re-run the model registration cell above.")

## 7. Running autonomous agent quality simulation

We execute `gbench --quality-only` to evaluate multi-turn autonomous agent workflows. On its first run, `gbench` fetches the pinned GemmaClaw commit and builds its local `gbench-quality` Docker image from it (several minutes, about 8 GB); later runs reuse the image. The suite runs in a container of that image against the Ollama `/v1` endpoint.

The cell below runs against the Ollama endpoint. Ollama does not report its context window, so we pass the Modelfile's `num_ctx` as `--max-model-len`; gbench declares it to GemmaClaw. To use a **vLLM** server instead, point `--remote-endpoint` at it and pass the matching tokenizer, e.g.:

```bash
python -m gbench --quality-only \
        --remote-endpoint http://127.0.0.1:8000/v1 \
        --tokenizer google/gemma-4-26B-A4B-it \
        --results-dir ./results_quality
```

With `--remote-endpoint`, `gbench` auto-detects the served model id from `/v1/models` (so `--models` is optional here).

In [ ]:
# Execute autonomous agent quality evaluation
!gbench --quality-only \
        --models gemma4-qat:4b \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --max-model-len 32768 \
        --results-dir ./results_quality

## 8. Analyzing multi-turn session recall and tool routing

`gbench --quality-only` writes its result to `<results-dir>/<timestamp>/quality/quality_<model>_<format>.json` (for the run above: `./results_quality/<timestamp>/quality/quality_<served-model>_remote-endpoint.json`; each run gets its own `<timestamp>` directory). We load that file to inspect how accurately the QAT model maintains long-context session recall across multiple agent turns and routes structured tool invocations.

In [ ]:
import glob, json, os

# Quality results land in <results-dir>/<timestamp>/quality/quality_<model>_<format>.json.
# The served-model tag can contain characters like ':', so glob for the file.
paths = sorted(glob.glob("./results_quality/*/quality/quality_*.json"), key=os.path.getmtime)
if not paths:
    print("No quality result file found under ./results_quality/<run>/quality/ - read the error at "
          "the end of the run cell's output above, fix it, then re-run that cell.")
else:
    result_path = paths[-1]
    print(f"Loading {result_path}")
    data = json.load(open(result_path))
    print(f"Model:      {data.get('model_name')} ({data.get('format')})")
    rate = data.get("pass_rate")
    rate_str = (f"{rate:.1f}%" if isinstance(rate, (int, float))
                else f"withheld ({data.get('pass_rate_withheld_reason')})")
    print(f"Pass rate:  {rate_str}  "
          f"({data.get('passed_scenarios')}/{data.get('total_scenarios')} scenarios passed)")
    print(f"Failed:     {data.get('failed_scenarios')}   Blocked: {data.get('blocked_scenarios')}")

## 9. Filtering specific capability scenarios

You can run a subset of the QA scenarios by passing their scenario ids (the `id:` of each scenario under `qa/scenarios/` in the GemmaClaw checkout, not file paths), for example `--scenarios memory-recall mcp-plugin-tools-call`, to focus on specific agent capabilities.

In [ ]:
!gbench --quality-only \
        --models gemma4-qat:4b \
        --scenarios memory-recall mcp-plugin-tools-call \
        --remote-endpoint http://localhost:11434/v1 \
        --tokenizer google/gemma-4-E4B-it \
        --max-model-len 32768 \
        --results-dir ./results_quality_filtered

## 10. Session cleanup and server shutdown

We unload the model, stop the Ollama server if this notebook started it, and remove the temporary Modelfile.

In [ ]:
import subprocess, os

# Unload the model, then stop the Ollama server only if this notebook started it
# (a server you were already running is left running).
subprocess.run(["ollama", "stop", "gemma4-qat:4b"], check=False)
if "OLLAMA_PROC" in globals():
    OLLAMA_PROC.terminate()
    OLLAMA_PROC.wait(timeout=30)
    print("Stopped the Ollama server this notebook started.")
else:
    print("Ollama server left running (it was not started by this notebook).")
if os.path.exists("Modelfile.qat"):
    os.remove("Modelfile.qat")
print("Session cleanup complete.")